# GPT-500M Training using streaming services (A10G Edition [AWS_g5.xlarge])

**Architecture:** Decoder-only GPT with multi-head causal self-attention  
**Scale:** ~505M parameters (28 layers × 1152 embd × 16 heads, GPT-2 BPE vocab)  
**Data:** `open-web-math/open-web-math` streamed from HuggingFace (~14.7B tokens)  
**Epochs:** Configurable (1–3 recommended); streaming with per-epoch shuffle  
**Target GPU:** A10G (24 GB VRAM - DDR6) — uses gradient checkpointing to fit the model  

<mark>[#Note: Needs Modification]</mark>
**T4 vs A100 changes (effective batch size is identical at 131,072 tokens/step):**  
- `micro_batch_size`: 8 → 4 (halves peak activation memory per step)  
- `grad_accum_steps`: 16 → 32 (doubles accumulation to compensate)  
- `TransformerBlock` uses `torch.utils.checkpoint` — recomputes activations during
  the backward pass instead of storing them, trading ~33% extra compute for ~60%
  less activation memory. This is what makes the 505M model fit in 15.6 GB.  

**Flow:** Mount Drive → Install deps → Define model → Stream & tokenize → Train → Generate  

Re-running from cell `7. Train` automatically resumes from the latest checkpoint if one exists.  
Epoch progress is tracked inside the checkpoint so resume is seamless across sessions.

---
### Parameter accounting
```
Token embedding   :  50257 × 1152  =   57.9M   (tied with lm_head — no extra cost)
Position embedding:   1024 × 1152  =    1.2M
Per transformer block (×28):
  CausalSelfAttn  :  4 × 1152²    =    5.31M   (c_attn 3n² + c_proj n², no bias)
  FeedForward     :  8 × 1152²    =   10.62M   (fc1 n→4n + fc2 4n→n, no bias)
  LayerNorms ×2   :  4 × 1152     =    4608
28 blocks total   :                 =  447.0M
Final LayerNorm   :                 =    2304
─────────────────────────────────────────────
TOTAL             :                 = ~505.1M
```

## 1. Setup & Environment

In [ ]:
import os
import getpass

# ── S3 (checkpoints & telemetry) ─────────────────────────────────────────────
S3_BUCKET  = "your-bucket-name"          # <-- edit: must already exist
S3_PREFIX  = "gpt500m"                    # <-- edit: S3 "folder" for this run
AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")   # <-- edit if needed
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

# ── AWS credentials (optional — prefer the instance's IAM role) ─────────────
if not os.environ.get("AWS_ACCESS_KEY_ID"):
    _aws_key = getpass.getpass(
        "AWS_ACCESS_KEY_ID (press Enter to use the instance's IAM role instead): "
    )
    if _aws_key:
        os.environ["AWS_ACCESS_KEY_ID"]     = _aws_key
        os.environ["AWS_SECRET_ACCESS_KEY"] = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
    del _aws_key

# ── Hugging Face token (optional) ───────────────────────────────────────────
if not os.environ.get("HF_TOKEN"):
    _hf_token = getpass.getpass("HF_TOKEN (optional — press Enter to skip): ")
    if _hf_token:
        os.environ["HF_TOKEN"] = _hf_token
    del _hf_token

print("Environment & credentials console configured:")
print(f"  AWS region      : {AWS_REGION}")
print(f"  AWS credentials : {'explicit keys' if os.environ.get('AWS_ACCESS_KEY_ID') else 'IAM role / instance profile'}")
print(f"  HF_TOKEN        : {'set' if os.environ.get('HF_TOKEN') else 'not set (public dataset access only)'}")
print(f"  S3 target       : s3://{S3_BUCKET}/{S3_PREFIX}/")


In [ ]:
!pip install -q datasets transformers tokenizers bitsandbytes pynvml boto3

In [ ]:
import os
import gc
import glob
import math
import time
from dataclasses import dataclass, field
from typing import Optional, Iterator
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as grad_ckpt
import csv
import shutil
import pynvml
import boto3
from botocore.exceptions import ClientError

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

def clear_gpu():
    global model, optimizer, scaler
    model     = None
    optimizer = None
    scaler    = None
    gc.collect()
    torch.cuda.empty_cache()

clear_gpu()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {device}")
if device == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
print(f"PyTorch : {torch.__version__}")
_NVML_OK = False
try:
    pynvml.nvmlInit()
    _NVML_OK = True
except Exception as _e:
    print(f"NVML init failed ({_e}); GPU utilization logging will report None.")


## 2. Mount Drive & Configure Paths

In [ ]:
# S3_BUCKET / S3_PREFIX / AWS_REGION are set in the Environment & Credentials Console above.

CKPT_DIR  = "/home/ec2-user/gpt500m/checkpoints"
CACHE_DIR = "/home/ec2-user/gpt500m/hf_cache"
os.makedirs(CKPT_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

s3 = boto3.client("s3")

def s3_key(*parts) -> str:
    return "/".join([S3_PREFIX, *parts])

def s3_upload_file(local_path: str, key: str):
    s3.upload_file(local_path, S3_BUCKET, key)

def s3_download_file(key: str, local_path: str) -> bool:
    """Returns False (no exception) if the key just doesn't exist yet —
    that's the expected/normal case on a genuinely fresh run."""
    try:
        s3.download_file(S3_BUCKET, key, local_path)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey"):
            return False
        raise

def s3_list_keys(prefix: str) -> list:
    paginator = s3.get_paginator("list_objects_v2")
    keys = []
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix):
        keys.extend(obj["Key"] for obj in page.get("Contents", []))
    return keys

# Sanity-check bucket access up front rather than failing deep into training
try:
    s3.head_bucket(Bucket=S3_BUCKET)
    print(f"S3 bucket OK   : s3://{S3_BUCKET}/{S3_PREFIX}/")
except ClientError as e:
    raise RuntimeError(
        f"Can't access bucket '{S3_BUCKET}' — check the name, your IAM "
        f"role/credentials, and that the bucket exists in this region."
    ) from e

print(f"Checkpoint dir : {CKPT_DIR}  (local; mirrored to S3)")
print(f"HF cache dir   : {CACHE_DIR}  (local only, not synced)")


## 3. Model Configuration

### Architecture
    # These numbers give ~505M parameters (see title cell for full breakdown).
    # Verified: 28 × (4×1152² + 8×1152² + 4×1152) + 50257×1152 + 1024×1152 + 2×1152
    vocab_size : int   = 50257   # GPT-2 BPE vocabulary (matches tiktoken/gpt2)
    block_size : int   = 1024    # context window length (tokens)
    n_layer    : int   = 28      # transformer depth
    n_head     : int   = 16      # attention heads; head_dim = 1152/16 = 72
    n_embd     : int   = 1152    # model width
    dropout    : float = 0.0     # 0.0 for large-scale pretraining (dropout hurts at scale)

    # ── Training ──────────────────────────────────────────────────────────────
    # OpenWebMath has ~14.7B tokens; 1 epoch ≈ 112k steps at batch=4, grad_accum=32
    # T4 config: micro_batch=4, grad_accum=32 → same 131,072 tokens/step as A100
    # (A100 used micro_batch=8, grad_accum=16 — identical effective batch, different split)
    num_epochs        : int   = 2       # 1–3 epochs recommended for math pretraining
    micro_batch_size  : int   = 4       # T4: 4 seqs × 1024 tokens = 4096 tokens/microstep
    grad_accum_steps  : int   = 32      # effective batch = 4×32×1024 = 131,072 tokens
    lr                : float = 3e-4   # peak learning rate (cosine decays to lr/10)
    weight_decay      : float = 0.1
    grad_clip         : float = 1.0
    warmup_steps      : int   = 1000   # ~1% of 1-epoch steps
    # train_steps computed dynamically from dataset size (see streaming section)
    train_steps       : int   = 112000 # overwritten at runtime
    eval_interval     : int   = 1000   # optimizer steps between eval + checkpoint
    eval_batches      : int   = 50     # mini-batches used to estimate loss

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size : int   = 50257
    block_size : int   = 1024
    n_layer    : int   = 28
    n_head     : int   = 16
    n_embd     : int   = 1152
    dropout    : float = 0.0

    # ── Training ──────────────────────────────────────────────────────────────
    num_epochs        : int   = 2
  
    micro_batch_size  : int   = 8
    grad_accum_steps  : int   = 16

    lr                : float = 3e-4
    weight_decay      : float = 0.1
    grad_clip         : float = 1.0
    warmup_steps      : int   = 1000

    train_steps       : int   = 112000
    eval_interval     : int   = 1000      # Changed from 100 to 10
    eval_batches      : int   = 50

    # ── Tokenizer ─────────────────────────────────────────────────────────────
    tokenizer_name    : str   = "gpt2"

    # ── Checkpoint retention ──────────────────────────────────────────────────
    max_checkpoints   : int   = 3

    # [FROM DOCS]
    # Gradient checkpointing trades ~33% extra compute for ~60% less activation
    # memory (see TransformerBlock). It was required to fit 505M params in
    # 15.6GB with the old fp32 Adam optimizer. The 8-bit optimizer + the larger
    # micro-batch above may free up enough headroom to disable it entirely —
    # that's not verifiable from here without a live T4, so it defaults to
    # True (safe/slower). Try flipping to False; if you hit CUDA OOM, set it
    # back to True.
    use_grad_checkpoint : bool  = True

    # Set at runtime
    device            : str   = "cuda" if torch.cuda.is_available() else "cpu"

cfg = GPTConfig()

assert cfg.n_embd % cfg.n_head == 0
print(f"Config Updated: Saving every {cfg.eval_interval} steps.")

## 4. Model Architecture

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head   = cfg.n_head
        self.n_embd   = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.dropout  = cfg.dropout

        # Fused Q+K+V projection: saves one kernel launch vs three Linears
        self.c_attn     = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.c_proj     = nn.Linear(cfg.n_embd, cfg.n_embd,     bias=False)
        self.resid_drop = nn.Dropout(cfg.dropout)

        # KV-cache tensors — None during training, populated during generation
        self._cache_k: Optional[torch.Tensor] = None
        self._cache_v: Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor, use_cache: bool = False) -> torch.Tensor:
        B, T, C = x.shape  # batch, seq len, channels (= n_embd)

        # Project and split: (B, T, 3C) → three (B, T, C)
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # Reshape to (B, n_head, T, head_dim) for batched attention
        def split_heads(t: torch.Tensor) -> torch.Tensor:
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        # ── KV-cache (inference only) ──────────────────────────────────────────
        if use_cache:
            if self._cache_k is not None:
                k = torch.cat([self._cache_k, k], dim=2)  # grow along seq dim
                v = torch.cat([self._cache_v, v], dim=2)
            self._cache_k = k
            self._cache_v = v

        # ── Scaled dot-product attention ──────────────────────────────────────
        # Dispatches to Flash Attention 2 kernel when available (PyTorch ≥ 2.0).
        # is_causal=True applies the causal mask efficiently without materialising
        # the full (T×T) attention matrix — crucial for long contexts.
        dropout_p = self.dropout if self.training else 0.0
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask  = None,
            dropout_p  = dropout_p,
            is_causal  = True,
        )

        # Merge heads: (B, n_head, T, head_dim) → (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

    def clear_cache(self):
        self._cache_k = None
        self._cache_v = None


class FeedForward(nn.Module):
    # [FROM DOCS]
    # Position-wise FFN: Linear(n_embd → 4n) → GELU → Linear(4n → n_embd).

    # The 4× expansion is the standard GPT ratio; it roughly doubles FLOPs per
    # token vs attention alone, and empirically provides most of the model's
    # 'memory' capacity (key-value storage in the FFN weights).
    # GELU is used (vs ReLU) because it is smooth and has non-zero gradient for
    # negative inputs, which aids gradient flow at initialisation.

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    # [FROM DOCS]
    # Pre-LayerNorm decoder block (GPT-2 style):
    #     x = x + Attn(LN(x))
    #     x = x + FFN(LN(x))

    # Pre-LN (vs post-LN as in the original Transformer) moves the normalisation
    # before the sublayer rather than after the residual add. This makes gradients
    # better conditioned and removes the need for a learning-rate warm-up as long,
    # though we keep warmup anyway for the lr schedule.

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.n_embd)
        self.ffn  = FeedForward(cfg)
        # Runtime toggle (was hardwired on before). See GPTConfig.use_grad_checkpoint
        # for the memory/compute tradeoff this controls.
        self.use_checkpoint = cfg.use_grad_checkpoint

    def _block_fn(self, x: torch.Tensor) -> torch.Tensor:
        """The core block computation, extracted so checkpoint() can wrap it."""
        x = x + self.attn(self.ln1(x), use_cache=False)
        x = x + self.ffn(self.ln2(x))
        return x

    def forward(self, x: torch.Tensor, use_cache: bool = False) -> torch.Tensor:
        if self.training and not use_cache and self.use_checkpoint:
            # Gradient checkpointing: recompute activations during backward.
            # use_reentrant=False is necessary when combined with torch.amp.autocast
            # (the reentrant variant does not propagate autocast context correctly).
            return grad_ckpt.checkpoint(self._block_fn, x, use_reentrant=False)
        # Inference / generation path, or checkpointing disabled: run normally
        x = x + self.attn(self.ln1(x), use_cache=use_cache)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT500M(nn.Module):
    """
    Decoder-only GPT with ~505M parameters.

    Key details:
    - Weight tying: tok_emb.weight == head.weight  (saves 50257×1152 ≈ 58M params
      and has been shown to regularise large LMs; Inan et al. 2017).
    - _init_weights applies the GPT-2 scheme:
        * Linear/Embedding: N(0, 0.02)
        * c_proj (the output projection of each sublayer) gets 1/√(2L) scaling
          so the contribution of each residual branch to the residual stream
          stays O(1) regardless of depth L (Press et al. 2022).
    """

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg      = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb  = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop     = nn.Dropout(cfg.dropout)
        self.blocks   = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_final = nn.LayerNorm(cfg.n_embd)
        self.head     = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        # Weight tying: the input and output token matrices are identical.
        # This halves the memory for the two largest parameter tensors.
        self.head.weight = self.tok_emb.weight

        self._init_weights()
        n = self.num_params()
        print(f"GPT-500M | {n:,} params ({n/1e6:.1f}M) | device: {cfg.device}")

    def _init_weights(self):
        for name, module in self.named_modules():
            if isinstance(module, nn.Linear):
                std = 0.02
                # Depth-scaled init for output projections of each residual branch:
                # σ = 0.02 / √(2 * n_layer).  Factor 2 because each block has two
                # sublayers (attn + ffn) each contributing to the residual stream.
                if name.endswith(("c_proj", "net.2")):
                    std = 0.02 / math.sqrt(2 * self.cfg.n_layer)
                nn.init.normal_(module.weight, mean=0.0, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def num_params(self) -> int:
        # Exclude head.weight because it is tied to tok_emb.weight
        return sum(p.numel() for n, p in self.named_parameters()
                   if not (n == "head.weight"))

    def forward(
        self,
        idx:       torch.Tensor,            # (B, T) int64
        targets:   Optional[torch.Tensor] = None,  # (B, T) int64
        use_cache: bool = False,
    ):
        B, T = idx.shape
        assert T <= self.cfg.block_size, \
            f"Sequence length {T} exceeds block_size {self.cfg.block_size}"

        positions = torch.arange(T, device=idx.device)  # (T,)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(positions))  # (B, T, C)

        for block in self.blocks:
            x = block(x, use_cache=use_cache)

        x      = self.ln_final(x)         # (B, T, C)
        logits = self.head(x)              # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Flatten batch × time into one dimension for cross-entropy
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )
        return logits, loss

    def clear_kv_cache(self):
        for block in self.blocks:
            block.attn.clear_cache()

    @torch.no_grad()
    def generate(
        self,
        prompt:         torch.Tensor,    # (1, T_prompt) int64
        max_new_tokens: int   = 256,
        temperature:    float = 0.8,
        top_k:          int   = 50,
        use_cache:      bool  = True,
    ) -> torch.Tensor:
        """Autoregressive generation with top-k sampling and optional KV-cache."""
        self.eval()
        self.clear_kv_cache()
        context   = prompt
        generated = []

        for step in range(max_new_tokens):
            # When using KV-cache, only the latest token is processed after step 0
            ctx = context[:, -self.cfg.block_size:]
            if use_cache and step > 0:
                ctx = context[:, -1:]

            logits, _ = self.forward(ctx, use_cache=use_cache)
            logits    = logits[:, -1, :] / temperature

            if top_k is not None and top_k > 0:
                topk_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < topk_vals[:, [-1]]] = float("-inf")

            probs      = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            context    = torch.cat([context, next_token], dim=1)
            generated.append(next_token.item())

        self.clear_kv_cache()
        return torch.tensor(generated)

## 5. Checkpoint Utilities

In [ ]:
def save_checkpoint(
    path:         str,
    model:        GPT500M,
    optimizer:    torch.optim.Optimizer,
    scaler:       torch.amp.GradScaler,
    step:         int,
    epoch:        int,
    epoch_step:   int,      # steps completed within current epoch
    losses:       dict,     # {"train": float, "val": float}
    cfg:          GPTConfig,
    ckpt_dir:     str,
    max_keep:     int = 3,
):
    """
    Save a full training snapshot.

    Stored keys:
      step         - global optimizer step (used for LR schedule on resume)
      epoch        - current epoch index (0-based)
      epoch_step   - steps done in this epoch (used to reposition the stream iterator)
      model        - model state_dict
      optimizer    - optimizer state_dict (includes momentum buffers)
      scaler       - GradScaler state (lost_scale history)
      cfg          - full GPTConfig (architecture + hyper-params)
      losses       - latest {train, val} loss estimates

    Rotation: only the `max_keep` most-recent checkpoints are retained on disk;
    older ones are deleted (oldest-first by modification time, same as reference).
    """
    os.makedirs(path if os.path.isdir(path) else os.path.dirname(path) or ".",
                exist_ok=True)

    ckpt_path = os.path.join(ckpt_dir, f"ckpt_{step:06d}.pt")
    torch.save({
        "step"       : step,
        "epoch"      : epoch,
        "epoch_step" : epoch_step,
        "model"      : model.state_dict(),
        "optimizer"  : optimizer.state_dict(),
        "scaler"     : scaler.state_dict(),
        "cfg"        : cfg,
        "losses"     : losses,
    }, ckpt_path)

    print(
        f"  checkpoint saved → {ckpt_path}  "
        f"(epoch {epoch} | step {step} | "
        f"train {losses['train']:.4f} | val {losses['val']:.4f})"
    )

    # ── Mirror to S3 (durability against instance loss / spot reclaim) ───────
    # This runs synchronously in the eval block (every eval_interval steps),
    # not the hot loop, so blocking on a ~6GB upload here is an acceptable
    # trade for "a reclaimed spot instance never loses more than one
    # eval_interval of progress."
    ckpt_key = s3_key("checkpoints", os.path.basename(ckpt_path))
    try:
        s3_upload_file(ckpt_path, ckpt_key)
        print(f"  → mirrored to s3://{S3_BUCKET}/{ckpt_key}")
    except Exception as e:
        # Never let an S3 hiccup kill a training run that's already
        # succeeded locally — surface it loudly and keep going.
        print(f"  ⚠ S3 upload FAILED ({e}); local checkpoint is still safe.")

    # ── Rotate: delete oldest checkpoints beyond max_keep (local AND S3) ─────
    all_ckpts = sorted(
        glob.glob(os.path.join(ckpt_dir, "ckpt_*.pt")),
        key=os.path.getmtime
    )
    for old_ckpt in all_ckpts[: max(0, len(all_ckpts) - max_keep)]:
        os.remove(old_ckpt)
        print(f"  deleted old checkpoint: {old_ckpt}")
        try:
            s3.delete_object(Bucket=S3_BUCKET, Key=s3_key("checkpoints", os.path.basename(old_ckpt)))
        except Exception as e:
            print(f"  ⚠ S3 delete of old checkpoint failed (non-fatal): {e}")


def load_checkpoint(path: str, device: str):
    """
    Restore model, optimizer, scaler, and all training state from a checkpoint.

    Two-phase load to avoid VRAM spike on T4:

    BROKEN (old): torch.load(map_location='cuda') loads the full checkpoint
    (~6.1 GB: params fp32 + Adam m/v) directly into VRAM, then GPT500M(cfg)
    allocates a second fresh model on GPU before load_state_dict runs — peak
    is ~8.5 GB before any forward pass, leaving almost no headroom.

    FIXED: deserialise to CPU (system RAM) first, allocate model skeleton on GPU,
    load_state_dict in-place (CPU->GPU copy, no second allocation), then move
    optimizer m/v buffers to GPU explicitly. Peak VRAM during load: ~2.4 GB.
    """
    print(f"Loading checkpoint: {path}")

    # Phase 1: deserialise entirely onto CPU RAM — zero VRAM cost
    ckpt = torch.load(path, map_location='cpu', weights_only=False)

    cfg = ckpt["cfg"]
    cfg.device = device

    # Phase 2: allocate model skeleton on GPU (~2.0 GB), then in-place copy
    # from CPU tensors into the existing GPU buffers — no second full allocation
    model = GPT500M(cfg).to(device)
    model = torch.compile(model)
    model.load_state_dict(ckpt["model"])
    del ckpt["model"]  # release CPU copy immediately to free RAM

    # Phase 3: reconstruct optimizer, restore state, then move m/v to GPU.
    # load_state_dict alone does NOT move tensors to GPU when the checkpoint
    # was loaded with map_location='cpu' — they stay on CPU, causing a device
    # mismatch crash on the first optimizer.step() after resume.
    decay_params    = [p for n, p in model.named_parameters()
                       if p.dim() >= 2 and p.requires_grad]
    no_decay_params = [p for n, p in model.named_parameters()
                       if p.dim() <  2 and p.requires_grad]
    import bitsandbytes as bnb
    OptClass = getattr(bnb.optim, "PagedAdamW8bit", None) or bnb.optim.AdamW8bit
    optimizer = OptClass([
        {"params": decay_params,    "weight_decay": cfg.weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=cfg.lr, betas=(0.9, 0.95))
    if "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
        for state in optimizer.state.values():  # move m/v buffers CPU -> GPU
            for k, v in state.items():
                if isinstance(v, torch.Tensor):
                    state[k] = v.to(device)
        del ckpt["optimizer"]  # release CPU copy

    # Phase 4: scaler (tiny — just a few Python scalars, no VRAM concern)
    scaler = torch.amp.GradScaler('cuda')
    if "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])

    gc.collect()  # ensure all remaining CPU-side ckpt tensors are freed

    step       = ckpt["step"]
    epoch      = ckpt.get("epoch",      0)
    epoch_step = ckpt.get("epoch_step", 0)
    losses     = ckpt["losses"]

    print(
        f"  resumed — global step {step} | epoch {epoch} | epoch_step {epoch_step}\n"
        f"  train loss {losses['train']:.4f} | val loss {losses['val']:.4f}\n"
        f"  VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated "
        f"/ {torch.cuda.memory_reserved()/1e9:.2f} GB reserved"
    )
    return model, optimizer, scaler, cfg, step, epoch, epoch_step, losses

## 6. Streaming Data Pipeline <mark>[#Note: Needs Modification]</mark>

We stream `open-web-math/open-web-math` directly from HuggingFace using
`datasets.load_dataset(..., streaming=True)`.  This means **no full download** — shards
are fetched on demand, making it practical even on Colab's limited disk.

The pipeline:
1. `TokenBuffer` wraps the HF streaming iterator, tokenises each document with
   `tiktoken` (GPT-2 BPE), appends an `<|endoftext|>` separator, and packs tokens
   into fixed-length `block_size` chunks.  This is the standard "pack and stride"
   approach — no padding, maximum GPU utilisation.
2. For evaluation we hold a **fixed in-memory validation shard** (first 500 documents
   of the validation split) to ensure comparable loss numbers across checkpoints.
3. Epoch bookkeeping: `epoch_step` in the checkpoint lets us `skip` already-seen
   examples within the current epoch on resume (HF streaming supports `.skip()`).

In [ ]:
import tiktoken
from datasets import load_dataset

# ── Tokenizer ────────────────────────────────────────────────────────────────
enc = tiktoken.get_encoding("gpt2")   # 50257-token BPE, matches cfg.vocab_size
EOT = enc.eot_token                   # 50256  — document separator
print(f"Tokenizer  : gpt2 (tiktoken)")
print(f"Vocab size : {enc.n_vocab}")
print(f"EOT token  : {EOT}")


class TokenBuffer:
    """
    Wraps a HuggingFace streaming dataset and yields fixed-length token chunks.

    Documents are tokenised on the fly, separated by <|endoftext|>, and packed
    into blocks of `block_size` tokens.  The last partial block of each document
    is held in `_buf` and completed by the next document — zero padding, zero waste.

    Usage:
        buf = TokenBuffer(stream_iter, block_size=1024)
        for chunk in buf:          # each chunk is a list[int] of length block_size
            ...
    """

    def __init__(self, iterator, block_size: int):
        self._iter       = iterator
        self._block_size = block_size
        self._buf: list[int] = []

    def __iter__(self):
        return self._generate()

    def _generate(self) -> Iterator[list[int]]:
        for doc in self._iter:
            # Tokenise and append EOT separator between documents
            tokens = enc.encode_ordinary(doc["text"]) + [EOT]
            self._buf.extend(tokens)

            # Emit as many full blocks as possible from the buffer
            while len(self._buf) >= self._block_size + 1:
                # +1 because we need block_size inputs AND block_size targets
                # (target[i] = input[i+1] — the next-token prediction)
                chunk = self._buf[: self._block_size + 1]
                self._buf = self._buf[self._block_size + 1 :]
                yield chunk    # list of block_size+1 ints


def make_stream(epoch: int, skip_docs: int = 0):
    ds = load_dataset(
        "open-web-math/open-web-math",
        split       = "train",
        streaming   = True,
        cache_dir   = CACHE_DIR,
        token       = os.environ.get("HF_TOKEN"),
    )
    ds = ds.shuffle(seed=42 + epoch, buffer_size=10_000)
    if skip_docs > 0:
        print(f"  fast-forwarding {skip_docs:,} documents in epoch {epoch} ...")
        ds = ds.skip(skip_docs)
    return iter(ds)


def make_val_tensors(n_val_docs: int = 500, block_size: int = 1024):
    print(f"Building validation set from {n_val_docs} val docs ...")
    val_ds  = load_dataset(
        "open-web-math/open-web-math",
        split     = "train",   # Using 'train' split as 'test' split is unavailable
        streaming = True,
        cache_dir = CACHE_DIR,
        token     = os.environ.get("HF_TOKEN"),
    )
    val_tokens = []
    for i, doc in enumerate(val_ds):
        if i >= n_val_docs:
            break
        val_tokens.extend(enc.encode_ordinary(doc["text"]) + [EOT])

    # Truncate to a multiple of block_size+1 and return as a 1D int64 tensor
    n = (len(val_tokens) // (block_size + 1)) * (block_size + 1)
    val_tensor = torch.tensor(val_tokens[:n], dtype=torch.int64)
    print(f"Validation tokens : {len(val_tensor):,}  ({len(val_tensor)/(block_size+1):.0f} blocks)")
    return val_tensor


# Build val set once (cached for the session)
val_data = make_val_tensors(n_val_docs=500, block_size=cfg.block_size)
print("Val set ready.")

## 7. Training Utilities

In [ ]:
def get_lr(step: int, cfg: GPTConfig) -> float:
    """
    Linear warmup → cosine decay schedule.

    - Warmup [0, warmup_steps): lr ramps linearly from 0 to cfg.lr.
      Warmup prevents very large gradient steps at init when the residual stream
      is not yet well-scaled.
    - Decay [warmup_steps, train_steps]: lr follows a half-cosine from cfg.lr
      down to cfg.lr * 0.1 (the Chinchilla/GPT-3 convention).
    - Uses the *global* step so the schedule is consistent across checkpoint resumes.
    """
    if step < cfg.warmup_steps:
        return cfg.lr * (step + 1) / cfg.warmup_steps
    progress = (step - cfg.warmup_steps) / max(1, cfg.train_steps - cfg.warmup_steps)
    # Cosine annealing: 1 → 0 over [0, 1]; scaled to [min_lr, lr]
    return cfg.lr * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))


def get_batch_from_stream(
    buf: TokenBuffer,
    micro_batch_size: int,
    block_size:       int,
    device:           str,
):
    """
    Draw `micro_batch_size` consecutive chunks from the token buffer.
    Returns x (inputs) and y (targets = x shifted right by 1).

    Each chunk has length block_size+1; we slice:
      x = chunk[:-1]   (tokens 0..block_size-1)
      y = chunk[1:]    (tokens 1..block_size  )
    so y[i] is the next-token target for x[i].
    """
    chunks = [next(buf._generate()) for _ in range(micro_batch_size)]
    data = torch.tensor(chunks, dtype=torch.int64)  # (B, block_size+1)
    x = data[:, :-1].to(device)
    y = data[:, 1: ].to(device)
    return x, y


@torch.no_grad()
def estimate_loss(
    model:    GPT500M,
    val_data: torch.Tensor,   # 1D token tensor on CPU (pre-built fixed val set)
    cfg:      GPTConfig,
) -> dict:
    """
    Estimate validation loss over `cfg.eval_batches` random windows from val_data.

    val_data lives on CPU to avoid occupying VRAM permanently. We move it to GPU
    once per eval call (not once per batch), sample random windows from it there,
    then delete the GPU copy when done — keeping VRAM usage to a single ~8 MB spike
    rather than 50 separate .to(device) round-trips.
    """
    model.eval()
    losses = []

    # Move val_data to GPU once for the duration of this eval pass
    val_gpu   = val_data.to(cfg.device)
    max_start = len(val_gpu) - cfg.block_size - 1

    with torch.amp.autocast('cuda'):
        for _ in range(cfg.eval_batches):
            ix = torch.randint(max_start, (cfg.micro_batch_size,))
            x  = torch.stack([val_gpu[i     : i + cfg.block_size    ] for i in ix])
            y  = torch.stack([val_gpu[i + 1 : i + cfg.block_size + 1] for i in ix])
            _, loss = model(x, y)
            losses.append(loss.item())

    del val_gpu  # release GPU copy; val_data (CPU) is retained for next eval
    torch.cuda.empty_cache()
    model.train()
    return {"val": sum(losses) / len(losses)}

## 8. Train

In [ ]:
# ── Telemetry helpers (Core Model Performance / Optimization Health /
#    System & Hardware Utilization / Data Throughput) ────────────────────────
#
# Design notes:
# - All of this is called ONLY at eval_interval (every 1000 steps), never
#   inside the inner micro-batch loop, so there is zero added overhead on the
#   ~999 steps out of every 1000 where none of this runs.
# - CSV is written LOCALLY first, then pushed to S3. Local-write-then-upload
#   means a network hiccup during the S3 push never corrupts the row that
#   was already safely on local disk — worst case is that one row's S3
#   mirror is stale until the next eval interval retries it.
# - Weight-norm and grad-norm are both "L2 norm of a collection of tensors";
#   torch.nn.utils.clip_grad_norm_ already computes exactly this for
#   gradients and returns it, so we just capture that return value rather
#   than recomputing it.

LOCAL_STATS_CSV_PATH = "/home/ec2-user/gpt500m/training_stats.csv"
STATS_CSV_S3_KEY     = s3_key("training_stats.csv")
STATS_CSV_LEGACY_S3_KEY = s3_key("training_stats.legacy.csv")

STATS_FIELDNAMES = [
    "step", "epoch",
    # Core Model Performance
    "train_loss", "train_perplexity", "val_loss", "val_perplexity",
    # Optimization Health
    "learning_rate", "grad_norm", "grad_clipping_ratio", "weight_norm",
    # System & Hardware Utilization
    "gpu_utilization_pct", "vram_allocated_gb", "vram_reserved_gb", "mfu_percentage",
    # Data Throughput
    "step_time", "tokens_per_sec",
]

# T4 Tensor Core peak throughput at fp16 (matches the autocast/GradScaler
# mixed-precision path this notebook trains in).
T4_PEAK_FLOPS = 65e12


def get_weight_norm(model: "GPT500M") -> float:
    """
    Global L2 norm of all trainable parameters:
        ||W|| = sqrt( sum_i ||W_i||_2^2 )
    Computed on-GPU, single .item() sync at the end — cheap even for 505M params.
    """
    with torch.no_grad():
        sq_sum = torch.zeros((), device=device)
        for p in model.parameters():
            if p.requires_grad:
                sq_sum += p.detach().float().pow(2).sum()
        return sq_sum.sqrt().item()


def get_gpu_snapshot() -> dict:
    """
    Point-in-time GPU utilization (NVML) + VRAM snapshot (PyTorch allocator).
    GPU util queried via NVML; returns None if NVML failed to init rather
    than raising, so a driver/permissions issue never interrupts training.
    """
    gpu_util = None
    if _NVML_OK:
        try:
            handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            gpu_util = pynvml.nvmlDeviceGetUtilizationRates(handle).gpu
        except Exception as _e:
            print(f"NVML query failed ({_e}); logging gpu_utilization_pct=None this interval.")

    return {
        "gpu_utilization_pct": gpu_util,
        "vram_allocated_gb": torch.cuda.memory_allocated() / 1e9,
        "vram_reserved_gb": torch.cuda.memory_reserved() / 1e9,
    }


def compute_mfu(tokens_per_sec: float, n_params: int) -> float:
    """
    Model FLOPs Utilization, PaLM/Chinchilla convention:
        FLOPs/token (fwd+backward, ideal) ≈ 6N
        MFU = tokens_per_sec * 6N / peak_hardware_FLOPs

    Note this deliberately uses the *ideal* 6N figure, not the extra ~33%
    of recompute FLOPs gradient checkpointing actually burns on this T4
    config — that's the point of MFU: it measures how much of peak hardware
    throughput turns into "useful" model FLOPs, so checkpointing correctly
    shows up as a lower MFU number even though it's a deliberate trade.
    """
    flops_per_token = 6 * n_params
    return 100.0 * (tokens_per_sec * flops_per_token) / T4_PEAK_FLOPS


def safe_perplexity(loss: float) -> float:
    """exp(loss) blows up to inf for loss > ~700; clamp to avoid OverflowError
    from a stray NaN/huge loss spiking the whole eval-row write."""
    try:
        return math.exp(min(loss, 20.0))
    except (OverflowError, ValueError):
        return float("inf")


def init_stats_csv():
    """
    Ensure a local CSV with the CURRENT schema exists before training starts.
      - Fresh run, nothing in S3 yet         -> new local file, write header.
      - Resuming, S3 file exists & schema OK -> download it so history (all
        prior eval rows) is preserved in the same file we keep appending to.
        This is what makes a spot-reclaim-and-relaunch resume correctly: the
        new instance has an empty local disk but pulls the real history down.
      - Resuming, S3 file exists but has the OLD 5-column schema (from
        before this telemetry upgrade) -> can't be safely appended to with
        DictWriter under the new fieldnames. Copied to
        training_stats.legacy.csv in S3 (never deleted) and a fresh file is
        started under the new schema.
    """
    if s3_download_file(STATS_CSV_S3_KEY, LOCAL_STATS_CSV_PATH):
        with open(LOCAL_STATS_CSV_PATH, newline="") as f:
            existing_header = next(csv.reader(f), [])
        if existing_header == STATS_FIELDNAMES:
            print(f"Resumed existing stats CSV from s3://{S3_BUCKET}/{STATS_CSV_S3_KEY}.")
            return
        else:
            s3.copy_object(Bucket=S3_BUCKET,
                            CopySource={"Bucket": S3_BUCKET, "Key": STATS_CSV_S3_KEY},
                            Key=STATS_CSV_LEGACY_S3_KEY)
            print(f"Old stats CSV schema detected — archived to "
                  f"s3://{S3_BUCKET}/{STATS_CSV_LEGACY_S3_KEY}, starting fresh.")

    with open(LOCAL_STATS_CSV_PATH, mode="w", newline="") as f:
        csv.DictWriter(f, fieldnames=STATS_FIELDNAMES).writeheader()
    print(f"Initialized fresh stats CSV at {LOCAL_STATS_CSV_PATH}.")


def write_stats_row(row: dict):
    """Append one row locally, then push the whole (small) file to S3 as a
    full overwrite — never an S3 append — so a half-written upload can never
    be the file a future resume reads back."""
    with open(LOCAL_STATS_CSV_PATH, mode="a", newline="") as f:
        csv.DictWriter(f, fieldnames=STATS_FIELDNAMES).writerow(row)
    try:
        s3_upload_file(LOCAL_STATS_CSV_PATH, STATS_CSV_S3_KEY)
    except Exception as e:
        print(f"⚠ S3 upload of stats CSV failed (non-fatal, local file still has this row): {e}")


In [ ]:
clear_gpu()

# Initialize a list to store training statistics
all_losses = []

# ── Resume or fresh start ────────────────────────────────────────────────────
# Two cases this needs to cover, not just "does a local checkpoint exist":
#   1. Same instance, notebook kernel restarted        -> local disk still
#      has checkpoints, nothing to download, just resume as before.
#   2. Fresh/replacement instance (e.g. spot reclaimed) -> local disk is
#      empty even though training progress exists — it's sitting in S3.
#      Pull the newest checkpoint down before falling through to the normal
#      "pick latest local checkpoint" logic below.
checkpoints = sorted(
    glob.glob(os.path.join(CKPT_DIR, "ckpt_*.pt")),
    key=os.path.getmtime
)
if not checkpoints:
    remote_ckpt_keys = sorted(k for k in s3_list_keys(s3_key("checkpoints"))
                               if k.endswith(".pt"))
    if remote_ckpt_keys:
        # ckpt_NNNNNN.pt is zero-padded, so lexicographic sort == numeric sort
        latest_key = remote_ckpt_keys[-1]
        local_path = os.path.join(CKPT_DIR, os.path.basename(latest_key))
        print(f"No local checkpoints found — pulling latest from "
              f"s3://{S3_BUCKET}/{latest_key} (likely a fresh/replacement instance)...")
        s3_download_file(latest_key, local_path)
        checkpoints = [local_path]
    else:
        print("No checkpoints found locally or in S3 — starting from scratch.")

latest_ckpt = checkpoints[-1] if checkpoints else None

if latest_ckpt:
    model, optimizer, scaler, cfg, start_step, start_epoch, start_epoch_step, loaded_losses = \
        load_checkpoint(latest_ckpt, device)

    # If resuming, add the loaded checkpoint's stats to all_losses
    all_losses.append({
        'step': start_step,
        'epoch': start_epoch,
        'train_loss': loaded_losses['train'],
        'val_loss': loaded_losses['val'],
        'lr': get_lr(start_step, cfg)
    })

    # load_checkpoint restores cfg exactly as it was pickled into the checkpoint,
    # which would silently overwrite any hyperparameter edits made in the config
    # cell (micro_batch_size, grad_accum_steps, use_grad_checkpoint) whenever
    # resuming from a checkpoint saved under the old settings. Re-apply this
    # run's throughput settings so config-cell edits always take effect, even
    # on resume — the model/optimizer/step state itself is unaffected.
    _fresh_cfg = GPTConfig()
    cfg.micro_batch_size    = _fresh_cfg.micro_batch_size
    cfg.grad_accum_steps    = _fresh_cfg.grad_accum_steps
    cfg.use_grad_checkpoint = _fresh_cfg.use_grad_checkpoint
    for _block in model.blocks:
        _block.use_checkpoint = cfg.use_grad_checkpoint
else:
    print("No checkpoint found — starting from scratch.")
    model     = GPT500M(cfg).to(device)
    model     = torch.compile(model)

    decay_params    = [p for n, p in model.named_parameters()
                       if p.dim() >= 2 and p.requires_grad]
    no_decay_params = [p for n, p in model.named_parameters()
                       if p.dim() <  2 and p.requires_grad]
    import bitsandbytes as bnb
    OptClass = getattr(bnb.optim, "PagedAdamW8bit", None) or bnb.optim.AdamW8bit
    optimizer = OptClass([
        {"params": decay_params,    "weight_decay": cfg.weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=cfg.lr, betas=(0.9, 0.95))
    scaler      = torch.amp.GradScaler('cuda')
    start_step  = 0
    start_epoch = 0
    start_epoch_step = 0

# ── Compute total steps across all epochs ────────────────────────────────────
DOCS_PER_EPOCH   = 6_300_000
STEPS_PER_EPOCH  = DOCS_PER_EPOCH * 350 // (cfg.block_size+1) // (cfg.micro_batch_size * cfg.grad_accum_steps)
cfg.train_steps  = STEPS_PER_EPOCH * cfg.num_epochs

print(f"\nSteps per epoch  : {STEPS_PER_EPOCH:,}")
print(f"Total steps      : {cfg.train_steps:,} ({cfg.num_epochs} epoch(s))")
print(f"Start step       : {start_step}")
print(f"Model params     : {model.num_params() / 1e6:.1f}M")

N_PARAMS = model.num_params()
init_stats_csv()

# ── Training loop ─────────────────────────────────────────────────────────────
model.train()
train_loss_acc = 0.0
t0 = time.time()

# Interval accumulators — reset every time we hit an eval boundary. These
# average out step-to-step noise in loss/step_time/grad_norm/tokens-sec so
# the CSV reflects a representative 1000-step window rather than one lucky
# (or unlucky) step.
interval_time       = 0.0
interval_loss       = 0.0
interval_grad_norm  = 0.0
interval_clip_count = 0
interval_steps      = 0

train_stream_iter = make_stream(epoch=start_epoch, skip_docs=start_epoch_step * cfg.grad_accum_steps * cfg.micro_batch_size)
train_buf = TokenBuffer(train_stream_iter, block_size=cfg.block_size)
buf_gen   = train_buf._generate()

epoch      = start_epoch
epoch_step = start_epoch_step

for step in range(start_step, cfg.train_steps):
    _step_t0 = time.time()  # wall-clock timer for this optimizer step (data + fwd + bwd + opt.step)

    if epoch_step >= STEPS_PER_EPOCH:
        epoch      += 1
        epoch_step  = 0
        if epoch >= cfg.num_epochs:
            print(f"\nCompleted {cfg.num_epochs} epoch(s). Training done.")
            break
        print(f"\n{'═'*60}\nStarting epoch {epoch}\n{'═'*60}")
        train_stream_iter = make_stream(epoch=epoch, skip_docs=0)
        train_buf = TokenBuffer(train_stream_iter, block_size=cfg.block_size)
        buf_gen   = train_buf._generate()

    lr = get_lr(step, cfg)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss_gpu = torch.zeros((), device=device)

    # Use a single autocast context for the entire accumulation loop to stabilize checkpointing
    with torch.amp.autocast('cuda'):
        for micro_step in range(cfg.grad_accum_steps):
            try:
                chunks = [next(buf_gen) for _ in range(cfg.micro_batch_size)]
            except StopIteration:
                epoch_step = STEPS_PER_EPOCH
                break

            data = torch.tensor(chunks, dtype=torch.int64).pin_memory()
            x = data[:, :-1].to(device, non_blocking=True)
            y = data[:, 1: ].to(device, non_blocking=True)

            _, loss = model(x, y)
            loss = loss / cfg.grad_accum_steps

            # Scale and backward call must be inside or correctly handle the autocast context
            scaler.scale(loss).backward()
            accum_loss_gpu += loss.detach()

    accum_loss = accum_loss_gpu.item()  # one sync per optimizer step, not one per micro-step

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    scaler.step(optimizer)
    scaler.update()

    epoch_step += 1
    train_loss_acc += accum_loss

    # ── Accumulate interval stats for this step ──────────────────────────────
    _step_time = time.time() - _step_t0
    interval_time      += _step_time
    interval_loss       += accum_loss
    interval_grad_norm  += grad_norm.item()
    if grad_norm.item() > cfg.grad_clip:
        interval_clip_count += 1
    interval_steps += 1

    if step % 10 == 0:
        smooth_loss = train_loss_acc / max(1, step - start_step + 1)
        print(f"ep {epoch} | step {step:>6d}/{cfg.train_steps} | loss {accum_loss:.4f} | smooth {smooth_loss:.4f} | lr {lr:.2e}", end='\r')

    if (step % cfg.eval_interval == 0) or (step == cfg.train_steps - 1):
        elapsed = time.time() - t0
        val_losses = estimate_loss(model, val_data, cfg)
        smooth_train = train_loss_acc / max(1, step - start_step + 1)
        losses = {"train": smooth_train, "val": val_losses["val"]}

        print(f"\n{'─'*65}\nEVAL  ep {epoch} | step {step:>6d} | train {losses['train']:.4f} | val {losses['val']:.4f} | {elapsed:.0f}s")

        save_checkpoint(path=CKPT_DIR, model=model, optimizer=optimizer, scaler=scaler, step=step, epoch=epoch, epoch_step=epoch_step, losses=losses, cfg=cfg, ckpt_dir=CKPT_DIR, max_keep=cfg.max_checkpoints)

        all_losses.append({'step': step, 'epoch': epoch, 'train_loss': losses['train'], 'val_loss': losses['val'], 'lr': lr})

        # ── Telemetry row (interval averages + point-in-time snapshots) ──────────
        _n = max(1, interval_steps)  # guard div-by-zero on a same-step double eval (e.g. step 0)
        avg_step_time  = interval_time / _n
        avg_train_loss = interval_loss / _n
        avg_grad_norm  = interval_grad_norm / _n
        clip_ratio     = interval_clip_count / _n

        tokens_per_sec = (cfg.micro_batch_size * cfg.grad_accum_steps * cfg.block_size) / avg_step_time if avg_step_time > 0 else 0.0
        mfu_pct        = compute_mfu(tokens_per_sec, N_PARAMS)
        w_norm         = get_weight_norm(model)
        gpu_snap       = get_gpu_snapshot()

        stats_row = {
            "step": step, "epoch": epoch,
            "train_loss": avg_train_loss, "train_perplexity": safe_perplexity(avg_train_loss),
            "val_loss": losses["val"], "val_perplexity": safe_perplexity(losses["val"]),
            "learning_rate": lr, "grad_norm": avg_grad_norm,
            "grad_clipping_ratio": clip_ratio, "weight_norm": w_norm,
            "gpu_utilization_pct": gpu_snap["gpu_utilization_pct"],
            "vram_allocated_gb": gpu_snap["vram_allocated_gb"],
            "vram_reserved_gb": gpu_snap["vram_reserved_gb"],
            "mfu_percentage": mfu_pct,
            "step_time": avg_step_time, "tokens_per_sec": tokens_per_sec,
        }
        write_stats_row(stats_row)
        print(f"STATS ep {epoch} | grad_norm {avg_grad_norm:.3f} | clip% {clip_ratio*100:.1f} | "
              f"tok/s {tokens_per_sec:,.0f} | MFU {mfu_pct:.1f}% | GPU {gpu_snap['gpu_utilization_pct']} | "
              f"VRAM {gpu_snap['vram_allocated_gb']:.2f}/{gpu_snap['vram_reserved_gb']:.2f} GB")

        # Reset interval accumulators for the next window
        interval_time = interval_loss = interval_grad_norm = 0.0
        interval_clip_count = interval_steps = 0

        t0 = time.time()
        model.train()

print("\nTraining complete.")

## 9. Generate

Loads the latest checkpoint, then autoregressively samples from the model.
Edit `PROMPT_TEXT`, `TEMPERATURE`, and `TOP_K` freely.

In [ ]:
# ── Load latest checkpoint for generation ────────────────────────────────────
checkpoints = sorted(
    glob.glob(os.path.join(CKPT_DIR, "ckpt_*.pt")),
    key=os.path.getmtime
)
if not checkpoints:
    # Same fresh-instance fallback as the training cell — pull from S3 if
    # this is a different instance than the one that trained the model.
    remote_ckpt_keys = sorted(k for k in s3_list_keys(s3_key("checkpoints")) if k.endswith(".pt"))
    if remote_ckpt_keys:
        latest_key = remote_ckpt_keys[-1]
        local_path = os.path.join(CKPT_DIR, os.path.basename(latest_key))
        print(f"Pulling latest checkpoint from s3://{S3_BUCKET}/{latest_key}...")
        s3_download_file(latest_key, local_path)
        checkpoints = [local_path]
assert checkpoints, "No checkpoints found locally or in S3 — run the training cell first."
gen_model, _, _, gen_cfg, gen_step, gen_epoch, _, _ = \
    load_checkpoint(checkpoints[-1], device)

# ── Generation config ────────────────────────────────────────────────────────
PROMPT_TEXT    = "The derivative of sin(x) with respect to x is"
MAX_NEW_TOKENS = 256
TEMPERATURE    = 0.8
TOP_K          = 50     # 0 to disable top-k filtering

# ── Encode prompt with tiktoken ───────────────────────────────────────────────
prompt_ids = enc.encode_ordinary(PROMPT_TEXT)
prompt_tensor = torch.tensor([prompt_ids], dtype=torch.int64).to(device)
print(f"Prompt : {repr(PROMPT_TEXT)}")
print(f"Tokens : {prompt_ids}")

# ── Generate ─────────────────────────────────────────────────────────────────
generated_ids = gen_model.generate(
    prompt         = prompt_tensor,
    max_new_tokens = MAX_NEW_TOKENS,
    temperature    = TEMPERATURE,
    top_k          = TOP_K,
    use_cache      = True,
)

# ── Decode ────────────────────────────────────────────────────────────────────
full_ids  = prompt_ids + generated_ids.tolist()
full_text = enc.decode(full_ids)

print("\n" + "─" * 65)
print(f"[Generated text — checkpoint step {gen_step} / epoch {gen_epoch}]")
print("─" * 65)
print(full_text)